In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/CV Bootcamp'
OUT = DATA + '/outputs'
import os, sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
os.makedirs(OUT, exist_ok=True)
sys.path.append(DATA)
from model import ResNet18
torch.manual_seed(0); np.random.seed(0)
dev = 'cuda'

Mounted at /content/drive


In [ ]:
X = torch.tensor(np.load(f'{DATA}/data/train_images.npy')).permute(0, 3, 1, 2).float().div(255)
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
X = ((X - mean) / std).to(dev)
lab = pd.read_csv(f'{DATA}/data/labels.csv').set_index('index').sort_index()
tr = np.loadtxt(f'{DATA}/data/train_idx.txt', dtype=int)
va = np.loadtxt(f'{DATA}/data/val_idx.txt', dtype=int)
noisy = torch.tensor(lab.noisy_label.values).to(dev)
clean = torch.tensor(lab.clean_label.values).to(dev)
tr_t, va_t = torch.tensor(tr).to(dev), torch.tensor(va).to(dev)

In [ ]:
def augment(x):
    n = x.size(0)
    x = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (n,), device=dev), torch.randint(0, 9, (n,), device=dev)
    r = torch.arange(32, device=dev)
    x = x[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
          (i[:, None] + r)[:, None, :, None], (j[:, None] + r)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    return x

@torch.no_grad()
def probs(idx):
    net.eval()
    with torch.autocast('cuda', dtype=torch.float16):
        return torch.cat([F.softmax(net(X[idx[k:k + 1000]]).float(), 1) for k in range(0, len(idx), 1000)])

In [ ]:
E, B = 60, 128
net = ResNet18(10).to(dev)
opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
P = np.zeros((E, len(tr), 10), dtype=np.float16)
for ep in range(E):
    net.train()
    perm = tr_t[torch.randperm(len(tr_t), device=dev)]
    for k in range(0, len(perm), B):
        b = perm[k:k + B]
        with torch.autocast('cuda', dtype=torch.float16):
            loss = F.cross_entropy(net(augment(X[b])), noisy[b])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    P[ep] = probs(tr_t).cpu().numpy()
    pv = probs(va_t).argmax(1)
    print(ep + 1, f'val_noisy {(pv == noisy[va_t]).float().mean().item():.4f}',
          f'val_clean {(pv == clean[va_t]).float().mean().item():.4f}')
np.save(f'{OUT}/t3A_train_probs.npy', P)

1 val_noisy 0.2740 val_clean 0.3522
2 val_noisy 0.3152 val_clean 0.3918
3 val_noisy 0.3694 val_clean 0.4752
4 val_noisy 0.3772 val_clean 0.4914
5 val_noisy 0.4068 val_clean 0.5456
6 val_noisy 0.4284 val_clean 0.5906
7 val_noisy 0.4352 val_clean 0.5856
8 val_noisy 0.4080 val_clean 0.5538
9 val_noisy 0.4578 val_clean 0.6178
10 val_noisy 0.4740 val_clean 0.6366
11 val_noisy 0.4484 val_clean 0.6264
12 val_noisy 0.5010 val_clean 0.6990
13 val_noisy 0.4156 val_clean 0.5714
14 val_noisy 0.4652 val_clean 0.6498
15 val_noisy 0.4754 val_clean 0.6720
16 val_noisy 0.5110 val_clean 0.7124
17 val_noisy 0.5210 val_clean 0.7284
18 val_noisy 0.5262 val_clean 0.7458
19 val_noisy 0.4832 val_clean 0.6538
20 val_noisy 0.5140 val_clean 0.7278
21 val_noisy 0.5038 val_clean 0.6880
22 val_noisy 0.5248 val_clean 0.7512
23 val_noisy 0.5240 val_clean 0.7270
24 val_noisy 0.5224 val_clean 0.7340
25 val_noisy 0.5326 val_clean 0.7620
26 val_noisy 0.5072 val_clean 0.7166
27 val_noisy 0.5288 val_clean 0.7454
28 val_noi

In [ ]:
pbar = P.astype(np.float32).mean(0)
y = lab.noisy_label.values[tr]
p_given = pbar[np.arange(len(tr)), y]
other = pbar.copy(); other[np.arange(len(tr)), y] = 0
S = np.log(other.max(1) + 1e-12) - np.log(p_given + 1e-12)
pd.DataFrame({'index': tr, 'score': S}).to_csv(f'{OUT}/t3_noise_scores.csv', index=False)
wrong = lab.noisy_label.values[tr] != lab.clean_label.values[tr]
k = wrong.sum()
print('k =', k, ' precision@k =', wrong[np.argsort(-S)[:k]].mean(), ' AUROC =', roc_auc_score(wrong, S))

k = 18109  precision@k = 0.8666408967916506  AUROC = 0.9569396062669016
